# Creación de Variables Temporales (LAG y Rolling) mediante SQL

Para proporcionar contexto temporal a los modelos de clasificación, se diseñaron variables con retraso (lag features) y agregaciones móviles (rolling features) a partir de las métricas meteorológicas y oceanográficas. El objetivo es que los algoritmos identifiquen anomalías basándose en la evolución previa de las variables y no solo en registros aislados.

In [17]:
import sqlite3
import pandas as pd

# 1. Ruta absoluta exacta reconstruida desde tu explorador de Windows
ruta_csv = r'C:\Users\naila\Downloads\final_project_elnino\data\dataset_elnino_final.csv'

try:
    # 2. Cargar el DataFrame desde el disco
    dataset_elnino_final = pd.read_csv(ruta_csv)
    print(f"✓ Dataset cargado con éxito en memoria. Filas detectadas: {len(dataset_elnino_final):,}")
    
    # 3. Conectar a la base de datos local (se creará/actualizará en la raíz)
    conexion = sqlite3.connect('proyecto_final_elnino.db')
    
    # 4. Guardar los datos en la tabla 'datos_el_nino'
    dataset_elnino_final.to_sql('datos_el_nino', conexion, if_exists='replace', index=False)
    print("✓ ¡Tabla 'datos_el_nino' creada y guardada con éxito en la base de datos!")
    
    # 5. Cerrar la conexión
    conexion.close()

except FileNotFoundError:
    print("⚠️ El archivo CSV no se encontró en la ruta especificada. Verifica si el nombre de usuario 'naila' es correcto.")
except Exception as e:
    print(f"⚠️ Ocurrió un error inesperado: {e}")

✓ Dataset cargado con éxito en memoria. Filas detectadas: 178,080
✓ ¡Tabla 'datos_el_nino' creada y guardada con éxito en la base de datos!


In [23]:
# Calculamos las variables temporales

def calcular_variables_temporales(db_path: str) -> pd.DataFrame:
    """Conecta a la base de datos SQLite, calcula las variables con retraso temporal (LAG)

    y agregaciones móviles (Rolling) por boya geográfica, y devuelve el DataFrame listo.
    """
    try:
        # 1. Establecer la conexión con la base de datos SQLite local
        conexion = sqlite3.connect(db_path)
        
        # 2. Definir la consulta estructurada con funciones de ventana (Window Functions)
        query = """
        SELECT 
            complete_date,
            latitude,
            longitude,
            zonal_winds,
            meridional_winds,
            sst AS sst_today,
            alert_anomaly,
            -- [LAG] Histórico de la temperatura de la superficie del mar de 7 registros atrás
            LAG(sst, 7) OVER(
                PARTITION BY latitude, longitude 
                ORDER BY complete_date
            ) AS sst_lag_7,
            -- [LAG] Histórico de la temperatura de la superficie del mar de 15 registros atrás
            LAG(sst, 15) OVER(
                PARTITION BY latitude, longitude 
                ORDER BY complete_date
            ) AS sst_lag_15,
            -- [ROLLING] Media móvil del viento zonal basada en una ventana deslizante de 15 días
            AVG(zonal_winds) OVER(
                PARTITION BY latitude, longitude 
                ORDER BY complete_date
                ROWS BETWEEN 14 PRECEDING AND CURRENT ROW
            ) AS avg_zonal_wind_15d
        FROM datos_el_nino;
        """
        
        # 3. Ejecutar la consulta SQL y volcar los resultados en un DataFrame de Pandas
        df_resultado = pd.read_sql_query(query, conexion)
        
        # 4. Limpieza de registros iniciales que contienen NaN debido al desplazamiento temporal
        df_limpio = df_resultado.dropna(subset=['sst_lag_7', 'sst_lag_15']).copy()
        
        print(f"✓ Variables y medias móviles calculadas. Registros útiles: {len(df_limpio):,}")
        return df_limpio

    except sqlite3.Error as e:
        print(f"Error al conectar o ejecutar la consulta en SQLite: {e}")
        return pd.DataFrame()
        
    finally:
        # Asegurar el cierre de la conexión si el objeto llegó a inicializarse
        if 'conexion' in locals():
            conexion.close()

# --- EJECUCIÓN DEL PIPELINE ---
# Llamada a la función con el nuevo nombre adaptado
df_modelado = calcular_variables_temporales('proyecto_final_elnino.db')

if not df_modelado.empty:
    print(df_modelado.head())

    # --- EJECUCIÓN Y GUARDADO DIRECTO ---
df_modelado = calcular_variables_temporales('proyecto_final_elnino.db')

if not df_modelado.empty:
    # Guardamos el resultado sobreescribiendo el dataset final para centralizar los datos
    df_modelado.to_csv('dataset_elnino_final.csv', index=False)
    print("✓ ¡Columnas reales guardadas directamente en 'dataset_elnino_final.csv'!")
       


✓ Variables y medias móviles calculadas. Registros útiles: 109,281
    complete_date  latitude  longitude  zonal_winds  meridional_winds  \
82     1996-03-06     -8.30    -154.95         -7.4              -2.1   
118    1993-06-29     -8.29    -154.99         -7.5              -1.3   
119    1993-06-30     -8.29    -154.99         -4.9              -2.8   
120    1993-07-18     -8.29    -154.99         -6.2              -3.0   
121    1993-07-20     -8.29    -154.99         -6.3               0.5   

     sst_today  alert_anomaly  sst_lag_7  sst_lag_15  avg_zonal_wind_15d  
82       28.26              0      28.13       29.76           -3.913333  
118      29.28              0      29.85       29.10           -4.580000  
119      29.28              0      29.84       29.47           -4.606667  
120      28.94              0      29.85       29.80           -4.713333  
121      28.90              0      29.49       29.55           -4.833333  
✓ Variables y medias móviles calculadas. Reg

In [24]:
df.head()

NameError: name 'df' is not defined